# 11c | Drills: Model Evaluation and Algorithms from Scratch

Covers **11a and 11b**: computing the confusion matrix and metrics by hand, changing the threshold, what AUC means, regression metrics, and the key steps of kNN, k-means, and gradient descent written in NumPy.

**How to use this lesson:** There are no input boxes; each problem is followed directly by its answer. For code questions, think about how you would write it first, then read the answer and its output. Questions marked 🗣 are oral: answer **out loud** in your own words first, then compare with the reference answer. After each one, cover the answer and try to restate the key points.

Code questions on this page run in order, so variables defined earlier can be reused later.

Every code question starts with this already run:

```python
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score
```

### Problem 1 · Confusion matrix by hand

Out of 10,000 transactions, 50 are fraudulent. The model flagged 200, of which 40 really are fraud. Compute the four cells and precision, recall, accuracy, and F1.

```python
tp = 40
fp = 200 - 40
fn = 50 - 40
tn = 10000 - tp - fp - fn
precision = tp / (tp + fp)
recall = tp / (tp + fn)
accuracy = (tp + tn) / 10000
f1 = 2 * precision * recall / (precision + recall)
print(tp, fp, fn, tn)
print(precision, recall, accuracy, round(f1, 4))
```

**Result:**

```text
40 160 10 9790
0.2 0.8 0.983 0.32
```

> Draw the 2×2 table first: 50 actual positives in total, 200 predicted positives in total. Accuracy is 98.3%, but precision is only 20%: only 1 in 5 alerts is real.

### Problem 2 · Counting the four cells with NumPy

Count TP, FP, FN, and TN from the 0/1 arrays `y_true` and `y_pred`.

```python
y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 0, 0])
y_pred = np.array([1, 0, 0, 1, 1, 0, 1, 0, 0, 1])
tp = np.sum((y_true == 1) & (y_pred == 1))
fp = np.sum((y_true == 0) & (y_pred == 1))
fn = np.sum((y_true == 1) & (y_pred == 0))
tn = np.sum((y_true == 0) & (y_pred == 0))
print(tp, fp, fn, tn)
```

**Result:**

```text
3 2 1 4
```

> Combine two Boolean arrays with `&`; `np.sum` counts the Trues.

### Problem 3 · Computing the metrics and checking against sklearn

Compute precision, recall, and F1 from the four counts above, then check against `precision_score` and `recall_score`.

```python
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
print(round(precision, 4), round(recall, 4), round(f1, 4))
print(round(precision_score(y_true, y_pred), 4), round(recall_score(y_true, y_pred), 4))
```

**Result:**

```text
0.6 0.75 0.6667
0.6 0.75
```

> Mnemonic: the denominator of precision is "predicted positive"; the denominator of recall is "actually positive." F1 is their harmonic mean and leans toward the smaller of the two.

### Problem 4 · Changing the threshold

For the same scores, how do precision and recall change between thresholds 0.5 and 0.3?

```python
scores = np.array([0.95, 0.85, 0.70, 0.60, 0.55, 0.40, 0.35, 0.30, 0.20, 0.10])
labels = np.array([1, 1, 0, 1, 0, 1, 0, 0, 0, 0])
for t in [0.5, 0.3]:
    pred = (scores >= t).astype(int)
    print(t, round(precision_score(labels, pred), 3), round(recall_score(labels, pred), 3))
```

**Result:**

```text
0.5 0.6 0.75
0.3 0.5 1.0
```

> Lowering the threshold flags more cases: recall rises and precision usually falls. The threshold is set by the costs of the two error types, not fixed at 0.5.

### Problem 5 · precision@k

You can only manually review the 3 highest-scoring cases. What fraction of them are true positives?

```python
top3 = np.argsort(-scores)[:3]
print(top3)
labels[top3].mean()
```

**Result:**

```text
[0 1 2]
```

```text
0.6667
```

> `np.argsort` gives indices in ascending order; sorting `-scores` gives descending order. When the business can only handle the top k, precision@k fits the need better than AUC.

### Problem 6 · AUC from the definition

AUC = the probability that a randomly chosen positive scores higher than a randomly chosen negative (ties count as 0.5). Write it with broadcasting, then check against `roc_auc_score`.

```python
def auc_score(y, s):
    pos = s[y == 1]
    neg = s[y == 0]
    greater = (pos[:, None] > neg[None, :]).sum()
    ties = (pos[:, None] == neg[None, :]).sum()
    return (greater + 0.5 * ties) / (len(pos) * len(neg))

print(auc_score(labels, scores), roc_auc_score(labels, scores))
```

**Result:**

```text
0.875 0.875
```

> Here 4 positives × 6 negatives = 24 pairs. This approach is O(#positives × #negatives); for large data, use sorting plus rank sums.

### Problem 7 · AUC only depends on ranking

If you divide the scores by 2, or cube them, does AUC change?

```python
print(roc_auc_score(labels, scores))
print(roc_auc_score(labels, scores / 2))
print(roc_auc_score(labels, scores ** 3))
```

**Result:**

```text
0.875
0.875
0.875
```

> Monotone transformations preserve the ranking, so AUC is unchanged. But after "dividing probabilities by 2" calibration is completely wrong, so a high AUC does not mean the probabilities are accurate.

### Problem 8 · 99% accuracy 🗣

In fraud detection, fraud is only 1% of cases, and the model's accuracy is 99%. What do you make of this? Which metrics should you look at?

**Reference answer:**

- **Interpretation:** flagging nothing (calling everything legitimate) also gives 99% accuracy, so this number carries no information.
- **What to look at:** recall (how much fraud is caught), precision (how many alerts are real), PR-AUC; if review capacity is limited, precision@k.
- **Also consider:** the costs of the two error types. What is the loss from a missed fraud versus wrongly blocking a legitimate transaction (customer experience, manual review cost)? Use them to set the threshold.
- **How to report:** report alongside the "call everything legitimate" baseline, and give the confusion matrix at the chosen threshold.

**Short version:** With 1% fraud, predicting "never fraud" already gets 99% accuracy, so I'd look at recall, precision, PR-AUC or precision at k, and set the threshold from the relative costs of missed fraud and false alarms.

### Problem 9 · Spam filtering 🗣

Should a spam filter prioritize precision or recall? Why?

**Reference answer:**

- **Prioritize precision:** sending a legitimate email (say, an interview invitation) to the spam folder costs far more than letting one spam email through.
- **Contrast with cancer screening:** there a missed diagnosis is costly, so recall matters more. The answer to "which metric" depends on the cost of each error type.
- **Approach:** choose the threshold that maximizes recall while keeping precision at or above some level (e.g., 99%).

**Short version:** Precision, because sending a legitimate email to spam is much costlier than letting some spam through. I'd pick the threshold that maximizes recall subject to a high precision floor.

### Problem 10 · ROC-AUC or PR-AUC 🗣

When do you use ROC-AUC, and when PR-AUC?

**Reference answer:**

- **ROC curve:** FPR on the x-axis, TPR on the y-axis. FPR's denominator is all negatives; when negatives are plentiful, even many false alarms move FPR only slightly, so the curve looks good.
- **PR curve:** recall on the x-axis, precision on the y-axis; it directly reflects "how many flagged cases are real" and is more sensitive when positives are rare.
- **Choice:** use ROC-AUC when classes are roughly balanced and you care about overall ranking; use PR-AUC when positives are rare and you care about the quality of the positives you find (fraud, rare diseases, recommendation clicks).
- **Different baselines:** a random model has ROC-AUC 0.5 and PR-AUC roughly equal to the positive rate.

**Short version:** ROC-AUC is fine for balanced problems, but with rare positives it looks optimistic because FPR has a huge denominator. PR-AUC focuses on precision among flagged cases, and its random baseline equals the positive rate.

### Problem 11 · RMSE vs. MAE with an outlier

One of 5 predictions is wildly off. Compute RMSE and MAE.

```python
y_reg = np.array([10.0, 12.0, 9.0, 11.0, 10.0])
y_hat = np.array([11.0, 11.0, 10.0, 12.0, 30.0])
err = y_reg - y_hat
rmse = np.sqrt(np.mean(err ** 2))
mae = np.mean(np.abs(err))
print(round(rmse, 3), round(mae, 3))
```

**Result:**

```text
8.989 4.8
```

> That error of 20 makes RMSE much larger than MAE. Minimizing MSE gives the conditional mean and minimizing MAE gives the conditional median, which is more robust to outliers.

### Problem 12 · R² by hand

Compute R² from `y_reg` and `y_hat` above.

```python
sse = np.sum((y_reg - y_hat) ** 2)
sst = np.sum((y_reg - y_reg.mean()) ** 2)
1 - sse / sst
```

**Result:**

```text
-76.6923
```

> R² = 1 − (model's sum of squared errors ÷ sum of squared errors of "always predict the mean"). When the model is worse than guessing the mean, R² is negative, as here.

### Problem 13 · Log loss punishes confident mistakes

Two sets of predictions have the same accuracy (at threshold 0.5 both get only the last one wrong). Compare their mean log loss.

```python
y_lab = np.array([1, 0, 1, 0])
p_mild = np.array([0.8, 0.3, 0.7, 0.6])
p_bold = np.array([0.8, 0.3, 0.7, 0.99])

def log_loss(y, p):
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

print(round(log_loss(y_lab, p_mild), 4), round(log_loss(y_lab, p_bold), 4))
```

**Result:**

```text
0.4632 1.3854
```

> The last negative is predicted at 0.99, and log loss penalizes it very heavily. Log loss measures the quality of the probabilities; accuracy only looks at right vs. wrong.

### Problem 14 · Why train with log loss? 🗣

Why are classification models trained with log loss instead of optimizing accuracy directly?

**Reference answer:**

- **Accuracy is not differentiable:** it is a step function; small changes in the parameters leave accuracy unchanged, so the gradient is zero almost everywhere and gradient descent can't be used.
- **Log loss is smooth:** it is convex for logistic regression, with a unique optimum; it is the negative log of the Bernoulli likelihood, so minimizing it is maximum likelihood estimation.
- **It cares about probabilities:** predicting 0.51 or 0.99 is the same for accuracy, but log loss distinguishes them, so the resulting probabilities are more meaningful (better calibrated).
- **Finally:** choose the threshold and evaluate accuracy after training, according to business needs.

**Short version:** Accuracy is a step function with zero gradient almost everywhere, while log loss is smooth, convex for logistic regression, and equals the negative Bernoulli log-likelihood, so it rewards good probabilities rather than just correct labels.

### Problem 15 · One kNN prediction by hand

With 6 training points in two classes, predict the class of point `[2, 2]` using k=3.

```python
X_tr = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [5.0, 5.0], [5.0, 6.0], [6.0, 5.0]])
y_tr = np.array([0, 0, 0, 1, 1, 1])
x = np.array([2.0, 2.0])
dist = np.sqrt(((X_tr - x) ** 2).sum(axis=1))
nearest = np.argsort(dist)[:3]
print(dist.round(2), nearest)
np.bincount(y_tr[nearest]).argmax()
```

**Result:**

```text
[2.83 2.24 2.24 4.24 5.   5.  ] [1 2 0]
```

```text
0
```

> Three steps: compute distances to all training points → `argsort` to take the k nearest → `bincount(...).argmax()` for the majority vote.

### Problem 16 · One round of k-means

With 4 points and 2 initial centers, run one round of "assign → update."

```python
pts = np.array([[1.0, 1.0], [1.5, 2.0], [8.0, 8.0], [9.0, 8.5]])
centers = np.array([[1.0, 1.0], [1.5, 2.0]])
d = ((pts[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
labels_km = d.argmin(axis=1)
centers = np.array([pts[labels_km == j].mean(axis=0) for j in range(2)])
print(labels_km)
print(centers)
```

**Result:**

```text
[0 1 1 1]
[[1.         1.        ]
 [6.16666667 6.16666667]]
```

> The initial centers are poorly chosen (both in the lower left). In the first round the second center gets 3 points and is pulled toward the upper right; after one more round, point (1.5, 2) returns to the first cluster and the two clusters separate. In practice, use k-means++ initialization and multiple runs.

### Problem 17 · The effect of the learning rate

Minimize f(w) = (w − 3)² with gradient descent, starting from w = 0 and taking 20 steps, and compare learning rates 0.1, 0.5, and 1.1.

```python
for lr in [0.1, 0.5, 1.1]:
    w = 0.0
    for _ in range(20):
        grad = 2 * (w - 3)
        w = w - lr * grad
    print(lr, round(w, 4))
```

**Result:**

```text
0.1 2.9654
0.5 3.0
1.1 -112.0128
```

> 0.1 slowly approaches 3; 0.5 gets there in one step (a special case for this function); 1.1 overshoots on every step by more and more, and diverges.

### Problem 18 · How do you choose a threshold? 🗣

The model outputs probabilities. How do you set the threshold for deployment?

**Reference answer:**

- **Don't default to 0.5:** 0.5 is reasonable only when the two error costs are equal and the probabilities are well calibrated.
- **By cost:** let the false-positive cost be c_FP and the false-negative cost c_FN; compute the total cost for each threshold on the validation set and take the minimum. For a well-calibrated model, the theoretically optimal threshold is c_FP / (c_FP + c_FN).
- **By constraint:** e.g., "precision at least 90%" or "at most 500 manual reviews per day"; among thresholds that satisfy the constraint, pick the one with the highest recall.
- **Where to choose it:** on validation-set or cross-validated predictions, not the test set; revisit it periodically after launch as the distribution changes.

**Short version:** I'd choose the threshold on validation data either by minimizing expected cost given the false positive and false negative costs, or by meeting an operational constraint such as a precision floor or review capacity.

### Problem 19 · It improved offline — ship it? 🗣

A new model raises offline AUC from 0.80 to 0.81. Should you ship it?

**Reference answer:**

- **First check reliability:** is the gain larger than the cross-validation noise; was the test set used only once; is there any new leakage?
- **Offline metrics are not the business goal:** a 0.01 AUC gain doesn't necessarily improve conversion or revenue; verify the business metric with an online A/B test.
- **Count the costs:** is the new model slower, harder to maintain, or dependent on a new data pipeline? Added latency can cancel out the gain.
- **Look at segments:** could an overall gain hide some user groups getting worse?
- **Conclusion:** the offline result justifies running an experiment; whether to ship depends on the experiment's primary and guardrail metrics.

**Short version:** A 0.01 AUC gain is a reason to run an online A/B test, not to ship directly. I'd check that it's beyond CV noise, weigh latency and maintenance costs, and decide on the experiment's business metrics and guardrails.

### Problem 20 · What do you monitor after launch? 🗣

Once the model is deployed, what would you monitor?

**Reference answer:**

- **Inputs:** each feature's distribution, missing rate, and value range, compared with training (catches upstream data errors or covariate shift).
- **Outputs:** the distribution of predicted scores and the share of positive predictions. A sudden shift in distribution often shows up earlier than a drop in performance.
- **Performance:** AUC, precision / recall, and calibration once labels come back; note that labels are delayed.
- **Business:** the business metrics the model serves (conversion, loss amounts) and guardrails.
- **Engineering:** latency, error rate, and feature-service timeouts (the default values used on timeout can quietly drag performance down).
- **Action:** set alert thresholds, retrain on a schedule or trigger, and revalidate after major changes.

**Short version:** I'd monitor input feature distributions and missingness, the score distribution, realized performance once labels arrive, the business metric, and serving health like latency and errors, with alerts and a retraining policy.

## Syntax recap

| Goal | Code |
|---|---|
| The four cells | `np.sum((y_true == 1) & (y_pred == 1))`, etc. |
| precision / recall / F1 | `tp / (tp + fp)`, `tp / (tp + fn)`, `2PR / (P + R)` |
| Changing the threshold | `(proba >= t).astype(int)` |
| Top k | `np.argsort(-scores)[:k]` |
| AUC by hand | `(pos[:, None] > neg[None, :]).sum()`, add 0.5 per tie, divide by number of pairs |
| RMSE / MAE / R² | `np.sqrt(np.mean(err ** 2))`, `np.mean(np.abs(err))`, `1 - SSE / SST` |
| log loss | `-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))` |
| kNN | distances → `argsort(dist)[:k]` → `bincount(...).argmax()` |
| k-means | `((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)` → mean of each cluster |
| Gradient descent | `w = w - lr * grad` |

The common thread in the oral questions: **choose metrics by error costs, choose thresholds on the validation set by cost, and confirm offline gains with an online experiment.**